# 04B Homework 04 ETM — Node Classification (Old Layout)

**Task 5 of the Final Assignment**

This notebook builds a **CellComplex** from the Brownstone floor plan room volumes, assigns
room-type labels and door-type apertures, exports the graph to CSV in the MSD model schema,
then runs the pretrained `msd_node_classifier.pt` to predict each room's type.

## Pipeline
1. Load room OBJs → build CellComplex with room-type labels  
2. Load door OBJs → add as apertures  
3. `Graph.ByTopology(cc, directApertures=True)` → circulation graph through doors  
4. Compute zoning and connectivity one-hot features per node/edge  
5. Export to CSV (MSD schema)  
6. Load with `PyG.ByCSVPath`, load pretrained model, predict  
7. Visualise true vs predicted labels

## MSD label → room-type mapping
| `room_type` | `label` | Zoning class |
|---|---:|---|
| `bedroom` | `0` | Private / static |
| `livingroom` | `1` | Living / dynamic |
| `kitchen` | `2` | Living / dynamic |
| `dining` | `3` | Living / dynamic |
| `corridor` | `4` | Living / dynamic |
| `stairs` | `5` | Service / functional |
| `storeroom` | `6` | Service / functional |
| `bathroom` | `7` | Service / functional |
| `balcony` | `8` | Outdoor / semi-outdoor |

**Room OBJs:** `Homework04/Objects/Old Layout/*.obj`  
**Apertures:** `door.obj` (door) · `Entrance door.obj` (entrance_door)  
**Reference:** instructor @channel pattern + HW02 `Cell.ByFaces` approach

## 1. Imports

In [20]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color
from topologicpy.PyG import PyG
import pandas as pd
import numpy as np
import os
from collections import Counter

## 2. Version check

In [21]:
print("This notebook requires topologicpy 0.9.43 or newer.")
print(Helper.Version())

This notebook requires topologicpy 0.9.43 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Renderer

In [22]:
renderer = "vscode"

## 4. Room-type and door-type mappings

### Zoning classes (for node features)
| Zoning | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | bedroom |
| Living/dynamic | `1` | livingroom, kitchen, dining, corridor |
| Service/functional | `2` | stairs, storeroom, bathroom |
| Outdoor/semi-outdoor | `3` | balcony |

### Connectivity classes (for node and edge features)
| Connection type | One-hot index | Meaning |
|---|---:|---|
| passage | `0` | Open opening / corridor connection |
| door | `1` | Standard interior door |
| entrance_door | `2` | Exterior / entrance door |

## 4. MSD label and feature mappings

In [23]:
ROOM_LABEL = {
    "bedroom": 0, "livingroom": 1, "kitchen": 2, "dining": 3,
    "corridor": 4, "stairs": 5, "storeroom": 6, "bathroom": 7, "balcony": 8,
}
ZONING = {
    "bedroom":   [1,0,0,0], "livingroom": [0,1,0,0], "kitchen":   [0,1,0,0],
    "dining":    [0,1,0,0], "corridor":   [0,1,0,0], "stairs":    [0,0,1,0],
    "storeroom": [0,0,1,0], "bathroom":   [0,0,1,0], "balcony":   [0,0,0,1],
}
NODE_CONNECTIVITY = {
    "bedroom":   [0,1,0], "livingroom": [0,1,0], "kitchen":  [0,1,0],
    "dining":    [0,1,0], "corridor":   [1,0,0], "stairs":   [1,0,0],
    "storeroom": [0,1,0], "bathroom":   [0,1,0], "balcony":  [0,1,0],
}
DOOR_CONNECTIVITY = {
    "passage":       [1,0,0],
    "door":          [0,1,0],
    "entrance_door": [0,0,1],
}

# Colors matching 02_Homework04_etm.ipynb exactly
ROOM_COLOR = {
    "bedroom":    "#FFBFBF",
    "bathroom":   "#4444FF",
    "corridor":   "#7FFFBF",
    "kitchen":    "#BF3F3F",
    "livingroom": "#FFBF00",
    "stairs":     "#BF3FFF",
    "storeroom":  "#FF7FFF",
    "dining":     "#A0522D",
    "balcony":    "#007F00",
    "unknown":    "#AAAAAA",
}
print("Mappings loaded.")

Mappings loaded.


## 5. Paths

In [24]:
\
OBJECTS_DIR  = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout"
MODEL_PATH   = r"C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt"
DATASET_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B_oldlayout"
os.makedirs(DATASET_PATH, exist_ok=True)
print("Objects :", OBJECTS_DIR)
print("Model   :", MODEL_PATH)
print("Dataset :", DATASET_PATH)

Objects : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout
Model   : C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt
Dataset : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B_oldlayout


## 6. Load room OBJs and build cells

Each OBJ file contains one room type (multiple objects = multiple floor levels).  
`Topology.ByOBJPath(transposeAxes=True)` converts Rhino Y-up → Z-up.  
`Cell.ByFaces` builds a solid cell at progressively loose tolerances (HW02 approach).

Each OBJ file represents one room type. We import the geometry, extract the enclosed cell
volumes, and create **selector vertices** (internal points) carrying `room_type`, `label`,
`cell_color`, `zoning`, and `connectivity` dictionaries. These selectors are later used to
transfer labels onto the merged CellComplex cells.

In [25]:
ROOM_FILES = {
    "Bedroom":     ("bedroom",    os.path.join(OBJECTS_DIR, "Bedroom.obj")),
    "Living room": ("livingroom", os.path.join(OBJECTS_DIR, "Living room.obj")),
    "Kitchen":     ("kitchen",    os.path.join(OBJECTS_DIR, "Kitchen.obj")),
    "Corridor":    ("corridor",   os.path.join(OBJECTS_DIR, "Corridor.obj")),
    "Stair":       ("stairs",     os.path.join(OBJECTS_DIR, "Stair.obj")),
    "Bathroom":    ("bathroom",   os.path.join(OBJECTS_DIR, "Bathroom.obj")),
    "Storeroom":   ("storeroom",  os.path.join(OBJECTS_DIR, "Storeroom.obj")),
    "Balcony":     ("balcony",    os.path.join(OBJECTS_DIR, "balcony.obj")),
    "Dining":      ("dining",     os.path.join(OBJECTS_DIR, "Dining.obj")),
}

def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

all_cells = []
selectors = []

for display_name, (room_type, obj_path) in ROOM_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    label  = ROOM_LABEL[room_type]
    zoning = ZONING[room_type]
    conn   = NODE_CONNECTIVITY[room_type]
    color  = ROOM_COLOR[room_type]

    n_ok = 0
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if len(faces) < 4:
            continue
        c = build_cell(faces)
        if c is None:
            continue

        d = Dictionary.ByKeysValues(
            ["room_type", "type", "label", "color", "cell_color",
             "feat_zoning_type_0", "feat_zoning_type_1",
             "feat_zoning_type_2", "feat_zoning_type_3",
             "feat_connectivity_0", "feat_connectivity_1",
             "feat_connectivity_2"],
            [room_type, display_name, label, color, color,
             zoning[0], zoning[1], zoning[2], zoning[3],
             conn[0], conn[1], conn[2]]
        )

        c  = Topology.SetDictionary(c, d)
        iv = Topology.InternalVertex(c)
        iv = Topology.SetDictionary(iv, d)
        selectors.append(iv)
        all_cells.append(c)
        n_ok += 1

    status = f"cells={n_ok}" if n_ok else "[SKIP] no cells built"
    print(f"  {display_name:20s} -> {room_type:12s}  label={label}  {status}")

print(f"\nTotal: {len(all_cells)} cells, {len(selectors)} selectors")

  Bedroom              -> bedroom       label=0  cells=7
  Living room          -> livingroom    label=1  cells=3
  Kitchen              -> kitchen       label=2  cells=1
  Corridor             -> corridor      label=4  cells=5
  Stair                -> stairs        label=5  cells=3
  Bathroom             -> bathroom      label=7  cells=2
  Storeroom            -> storeroom     label=6  cells=6
  Balcony              -> balcony       label=8  cells=2
  Dining               -> dining        label=3  cells=1

Total: 30 cells, 30 selectors


## 7. Build CellComplex and transfer room-type dictionaries

All room cells are merged into a single `CellComplex`. Then
`Topology.TransferDictionariesBySelectors` assigns the room-type dictionaries from the
selector vertices to the CellComplex cells.

In [26]:
cc = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
print("Topology type :", Topology.TypeAsString(cc))
print("Cells :", len(Topology.Cells(cc) or []))
print("Faces :", len(Topology.Faces(cc) or []))

# Progressive tolerance: start tight (avoids wrong assignments), widen only if needed.
# Balcony cells are spatially isolated so higher tolerance stays safe.
for _tol in [0.1, 0.5, 1.0, 2.0]:
    cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True, tolerance=_tol)
    _labelled = sum(1 for c in (Topology.Cells(cc) or [])
                    if Dictionary.ValueAtKey(Topology.Dictionary(c), "room_type"))
    print(f"  TransferDictionaries tolerance={_tol} → {_labelled}/{len(Topology.Cells(cc) or [])} cells labelled")
    if _labelled >= len(selectors):
        break

# Flatten any list-valued dictionary entries: TransferDictionariesBySelectors can wrap
# values in single-element lists, which makes room_type unhashable / unusable downstream.
def _flatten_topology_dict(topo):
    dd = Topology.Dictionary(topo)
    if dd is None:
        return
    nk, nv, changed = [], [], False
    for k in (Dictionary.Keys(dd) or []):
        v = Dictionary.ValueAtKey(dd, k)
        if isinstance(v, list):
            changed = True
            v = v[0] if v else None
        if v is not None:
            nk.append(k); nv.append(v)
    if changed and nk:
        Topology.SetDictionary(topo, Dictionary.ByKeysValues(nk, nv))

for cell in (Topology.Cells(cc) or []):
    _flatten_topology_dict(cell)

counts, unlabelled = Counter(), 0
for cell in (Topology.Cells(cc) or []):
    d  = Topology.Dictionary(cell)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if isinstance(rt, list):
        rt = rt[0] if rt else None
    if rt:
        counts[rt] += 1
    else:
        unlabelled += 1

print("\nRoom distribution:")
for rt, n in sorted(counts.items(), key=lambda x: ROOM_LABEL.get(x[0], 99)):
    print(f"  {rt:15s}  label={ROOM_LABEL[rt]}  count={n}")
if unlabelled:
    print(f"  *** {unlabelled} unlabelled cell(s) (SelfMerge artifacts — expected)")
else:
    print("  All cells labelled.")

Topology type : Cluster
Cells : 30
Faces : 200
  TransferDictionaries tolerance=0.1 → 30/30 cells labelled

Room distribution:
  bedroom          label=0  count=7
  livingroom       label=1  count=3
  kitchen          label=2  count=1
  dining           label=3  count=1
  corridor         label=4  count=5
  stairs           label=5  count=3
  storeroom        label=6  count=6
  bathroom         label=7  count=2
  balcony          label=8  count=2
  All cells labelled.


## 8. Visualise CellComplex coloured by room type

In [27]:
# Build display copies: RemoveCoplanarFaces removes OBJ triangulation visually,
# then copy the color dictionary back (the cleaned topology is a new object).
# all_cells stays untouched — SelfMerge and AddApertures need the original geometry.
display_cells = []
for c in all_cells:
    c2 = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
    if c2:
        c2 = Topology.RemoveCollinearEdges(c2) or c2
        c2 = Topology.SetDictionary(c2, Topology.Dictionary(c))
    display_cells.append(c2 if c2 else c)

Topology.Show(
    display_cells,
    selectors,
    faceColorKey="color",
    faceOpacity=0.4,
    showEdges=True, edgeWidth=3,
    showVertices=True, vertexSize=10,
    vertexLabelKey="type",
    showVertexLabel=True,
    backgroundColor="white",
    width=800, height=600,
    renderer=renderer
)

## 9. Load door OBJs as apertures

Doors are modelled as planar face objects. We collect faces from three door files:
- `doors2.obj` — standard interior doors (type `door`)
- `Entrance door.obj` — exterior / entrance doors (type `entrance_door`)
- `Passage Door.obj` — open passage connections (type `passage`)

Each face is tagged with a `door_type` dictionary, then passed as apertures to
`Topology.AddApertures`. When `Graph.ByTopology(cc, directApertures=True)` is called,
it creates graph edges only between cells whose shared face carries an aperture.

In [28]:
DOOR_FILES = {
    "door":          os.path.join(OBJECTS_DIR, "door.obj"),
    "entrance_door": os.path.join(OBJECTS_DIR, "Entrance door.obj"),
}

apertures = []

for door_type, obj_path in DOOR_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    conn = DOOR_CONNECTIVITY[door_type]
    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    faces_for_type = []
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if faces:
            faces_for_type.extend(faces)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is None:
                    w2 = Topology.RemoveCollinearEdges(w)
                    f  = Face.ByWire(w2) if w2 else None
                if f is not None:
                    faces_for_type.append(f)

    for f in faces_for_type:
        d = Dictionary.ByKeysValues(
            ["door_type",
             "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
            [door_type, conn[0], conn[1], conn[2]]
        )
        f = Topology.SetDictionary(f, d)
        apertures.append(f)

    print(f"  {door_type:20s} -> {len(faces_for_type)} aperture faces")

print(f"\nTotal apertures: {len(apertures)}")

  door                 -> 32 aperture faces
  entrance_door        -> 3 aperture faces

Total apertures: 35


## 10. Add apertures to CellComplex

In [29]:
if not apertures:
    print("No apertures found — graph will use direct cell adjacency.")
else:
    best_cc      = cc
    best_matched = 0

    for tol in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0]:
        cc_try  = Topology.AddApertures(
            cc, apertures, exclusive=False, subTopologyType="Face", tolerance=tol)
        matched = sum(
            1 for f in (Topology.Faces(cc_try) or [])
            if Topology.Apertures(f)
        )
        print(f"  tolerance={tol:5.3f}  ->  {matched}/{len(apertures)} faces matched")
        if matched > best_matched:
            best_matched = matched
            best_cc      = cc_try
        if matched == len(apertures):
            break

    cc = best_cc

    # Diagnose unmatched apertures: compare each aperture centroid to nearest CC face centroid
    if best_matched < len(apertures):
        print("\n--- Unmatched aperture diagnosis ---")
        cc_faces = Topology.Faces(cc) or []
        cc_face_centroids = []
        for f in cc_faces:
            vs = Topology.Vertices(f) or []
            if vs:
                cx = sum(Vertex.X(v) for v in vs) / len(vs)
                cy = sum(Vertex.Y(v) for v in vs) / len(vs)
                cz = sum(Vertex.Z(v) for v in vs) / len(vs)
                cc_face_centroids.append((cx, cy, cz))

        # Find matched aperture centroids (faces that have apertures)
        matched_centroids = set()
        for f in cc_faces:
            aps = Topology.Apertures(f) or []
            for ap in aps:
                if ap is None:
                    continue
                vs = Topology.Vertices(ap) or []
                if vs:
                    cx = round(sum(Vertex.X(v) for v in vs) / len(vs), 2)
                    cy = round(sum(Vertex.Y(v) for v in vs) / len(vs), 2)
                    cz = round(sum(Vertex.Z(v) for v in vs) / len(vs), 2)
                    matched_centroids.add((cx, cy, cz))

        def centroid(ap_face):
            vs = Topology.Vertices(ap_face) or []
            if not vs:
                return None
            return (
                sum(Vertex.X(v) for v in vs) / len(vs),
                sum(Vertex.Y(v) for v in vs) / len(vs),
                sum(Vertex.Z(v) for v in vs) / len(vs),
            )

        def nearest_cc_face_dist(pos):
            best = float("inf")
            for fc in cc_face_centroids:
                d = ((pos[0]-fc[0])**2 + (pos[1]-fc[1])**2 + (pos[2]-fc[2])**2)**0.5
                if d < best:
                    best = d
            return best

        for ap in apertures:
            c = centroid(ap)
            if c is None:
                continue
            cr = (round(c[0], 2), round(c[1], 2), round(c[2], 2))
            if cr not in matched_centroids:
                dist = nearest_cc_face_dist(c)
                dt   = Dictionary.ValueAtKey(Topology.Dictionary(ap), "door_type") or "?"
                print(f"  UNMATCHED  door_type={dt}  pos=({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})  "
                      f"nearest_face_dist={dist:.4f}")

    faces_with_ap = best_matched
    unmatched     = len(apertures) - faces_with_ap
    print(f"\nFaces carrying apertures : {faces_with_ap}")
    if unmatched:
        print(f"Unmatched apertures      : {unmatched}")

  tolerance=0.001  ->  30/35 faces matched
  tolerance=0.010  ->  30/35 faces matched
  tolerance=0.050  ->  30/35 faces matched
  tolerance=0.100  ->  30/35 faces matched
  tolerance=0.500  ->  36/35 faces matched
  tolerance=1.000  ->  36/35 faces matched
  tolerance=2.000  ->  48/35 faces matched

Faces carrying apertures : 48
Unmatched apertures      : -13


## 11. Build the room circulation graph

`Graph.ByTopology` with `directApertures=True` creates:
- One **vertex** per cell (room)  
- One **edge** between two cells whose shared face carries a door/passage aperture

Edges therefore represent **navigable connections** (movement through doors, passages, and
entrance doors), not raw wall adjacency — this is a circulation graph, mirroring the MSD
dataset construction. Same-floor filtering, the isolation patch, and the explicit
balcony/stair/kitchen links below all refine that circulation structure.

In [30]:
# Old-Layout circulation graph — proximity-based.
# The Old-Layout room volumes are SEPARATE solids that do not share faces, so
# Graph.ByTopology(cc, directApertures=True) produces no edges. Instead we mirror 02B:
#   * one graph vertex per room selector (already carry label + zoning/connectivity feats)
#   * edges from aperture-to-room proximity matching (real doors / entrance doors)
#   * explicit Old-Layout circulation links (entrance stair, large stair, top living room)
SAME_FLOOR_Z_TOL = 0.60

def _get_rt(v):
    val = Dictionary.ValueAtKey(Topology.Dictionary(v), "room_type")
    if isinstance(val, list):
        val = val[0] if val else None
    return val
def _adt(a):
    val = Dictionary.ValueAtKey(Topology.Dictionary(a), "door_type")
    if isinstance(val, list):
        val = val[0] if val else None
    return val
def _d3(a, b):
    return sum((x-y)**2 for x, y in zip(a, b))

areas = [Cell.SurfaceArea(c) for c in all_cells]

# Per-room face planes + bounding boxes (for aperture matching).
cell_face_pb = []
for c in all_cells:
    fd = []
    for f in (Topology.Faces(c) or []):
        fc = Topology.Centroid(f)
        fp = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn = Face.Normal(f)
        vs = Topology.Vertices(f) or []
        if vs:
            xs = [Vertex.X(v) for v in vs]; ys = [Vertex.Y(v) for v in vs]; zs = [Vertex.Z(v) for v in vs]
            bb = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bb = None
        fd.append((fp, fn, bb))
    cell_face_pb.append(fd)

def _perp(pt, fp, fn):
    return abs(fn[0]*(pt[0]-fp[0]) + fn[1]*(pt[1]-fp[1]) + fn[2]*(pt[2]-fp[2]))
def _in_bbox(pt, bb, m=0.10):
    if bb is None: return True
    return (bb[0]-m <= pt[0] <= bb[1]+m and bb[2]-m <= pt[1] <= bb[3]+m and bb[4]-m <= pt[2] <= bb[5]+m)

# Match each aperture to the room(s) it separates (centroid near a room face plane + bbox).
PLANE_TOL = 0.10
apt_to_rooms, apt_types = [], []
for a in apertures:
    ac = Topology.Centroid(a)
    ap = (Vertex.X(ac), Vertex.Y(ac), Vertex.Z(ac))
    rooms = set()
    for ri, fd in enumerate(cell_face_pb):
        for fp, fn, bb in fd:
            if _perp(ap, fp, fn) < PLANE_TOL and _in_bbox(ap, bb):
                rooms.add(ri); break
    apt_to_rooms.append(rooms); apt_types.append(_adt(a))

matched = sum(1 for r in apt_to_rooms if len(r) >= 2)
print(f"Apertures: {len(apertures)}  matched to >=2 rooms: {matched}")

def _rt(i): return _get_rt(selectors[i])
def _z(i):  return Vertex.Z(selectors[i])
def _xyz(i): return (Vertex.X(selectors[i]), Vertex.Y(selectors[i]), Vertex.Z(selectors[i]))

# Edge set: cell-index pair -> door_type
edge_pairs = {}
def _put(i, j, dt):
    if i != j:
        edge_pairs[(min(i, j), max(i, j))] = dt

# 1. Door / entrance-door access pairs (same level, or stair involved)
for ai, rooms in enumerate(apt_to_rooms):
    dt = apt_types[ai]
    if dt not in ("door", "entrance_door"):
        continue
    rl = sorted(rooms)
    for a in range(len(rl)):
        for b in range(a + 1, len(rl)):
            i, j = rl[a], rl[b]
            if _rt(i) != "stairs" and _rt(j) != "stairs" and abs(_z(i) - _z(j)) > SAME_FLOOR_Z_TOL:
                continue
            _put(i, j, dt)

# 2. Identify the two stairs by plan (X,Y) clustering (largest cluster = main/large)
stair_idx = [i for i in range(len(selectors)) if _rt(i) == "stairs"]
_TOL = 5.0
def _close(a, b):
    return (Vertex.X(selectors[a]) - Vertex.X(selectors[b]))**2 + (Vertex.Y(selectors[a]) - Vertex.Y(selectors[b]))**2 <= _TOL**2
_cl = []
for i in stair_idx:
    for c in _cl:
        if any(_close(i, j) for j in c):
            c.append(i); break
    else:
        _cl.append([i])
_ch = True
while _ch:
    _ch = False
    for a in range(len(_cl)):
        for b in range(a + 1, len(_cl)):
            if any(_close(i, j) for i in _cl[a] for j in _cl[b]):
                _cl[a] += _cl[b]; del _cl[b]; _ch = True; break
        if _ch: break
_cl.sort(key=len, reverse=True)
main_set = set(_cl[0]) if _cl else set()              # large / main stair
ext_set  = set(i for c in _cl[1:] for i in c)         # entrance stair

# 3. Explicit Old-Layout circulation links (mirrors 02B §15)
balc = [i for i in range(len(selectors)) if _rt(i) == "balcony"]
corr = [i for i in range(len(selectors)) if _rt(i) == "corridor"]
livg = [i for i in range(len(selectors)) if _rt(i) == "livingroom"]
beds = [i for i in range(len(selectors)) if _rt(i) == "bedroom"]

ext_cent = None
if ext_set:
    ext_cent = (sum(Vertex.X(selectors[i]) for i in ext_set)/len(ext_set),
                sum(Vertex.Y(selectors[i]) for i in ext_set)/len(ext_set),
                sum(Vertex.Z(selectors[i]) for i in ext_set)/len(ext_set))

small_balcony  = min(balc, key=lambda i: areas[i]) if balc else None      # smallest balcony
front_corridor = (min(corr, key=lambda i: _d3(_xyz(i), ext_cent)) if (corr and ext_cent) else None)
top_living     = max(livg, key=lambda i: _z(i)) if livg else None
top_beds = []
if top_living is not None:
    tz = _z(top_living)
    top_beds = [i for i in beds if abs(_z(i) - tz) <= 1.5]

# entrance stair ↔ front corridor + small balcony
for e in ext_set:
    if front_corridor is not None: _put(front_corridor, e, "door")
    if small_balcony  is not None: _put(small_balcony,  e, "entrance_door")
# large stair ↔ top living room (nearest large-stair cell); top living ↔ each top bedroom
if top_living is not None and main_set:
    msel = min(main_set, key=lambda s: _d3(_xyz(s), _xyz(top_living)))
    _put(top_living, msel, "door")
    for b in top_beds:
        _put(top_living, b, "door")
# constraint: small balcony must NOT connect to the large stair
if small_balcony is not None:
    for s in main_set:
        edge_pairs.pop((min(small_balcony, s), max(small_balcony, s)), None)

# 4. Build the graph: vertices = room selectors, edges tagged with connectivity features
sel_edges = []
for (i, j), dt in edge_pairs.items():
    e = Edge.ByVertices([selectors[i], selectors[j]])
    if not e:
        continue
    conn = DOOR_CONNECTIVITY[dt]
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(
        ["door_type", "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
        [dt, conn[0], conn[1], conn[2]]))
    sel_edges.append(e)

graph = Graph.ByVerticesEdges(selectors, sel_edges)

print(f"Graph: {len(Graph.Vertices(graph) or [])} room nodes, {len(Graph.Edges(graph) or [])} edges")
print(f"  entrance stair cells {sorted(ext_set)} ↔ front corridor "
      f"{front_corridor}, small balcony {small_balcony}")
print(f"  large stair cells {sorted(main_set)} ↔ top living room {top_living}; "
      f"top bedrooms {sorted(top_beds)}")
iso = [i for i in range(len(selectors)) if not any(i in p for p in edge_pairs)]
if iso:
    print(f"  note: {len(iso)} room(s) with no edge: {[(_rt(i)) for i in iso]}")

Apertures: 35  matched to >=2 rooms: 35
Graph: 30 room nodes, 32 edges
  entrance stair cells [16] ↔ front corridor 12, small balcony 27
  large stair cells [17, 18] ↔ top living room 9; top bedrooms [3, 4, 5, 6]


## 11B. Visualise the room circulation graph

In [31]:
# Two-colour edge scheme: edges that touch a balcony are yellow, everything else is gray.
BALCONY_EDGE_COLOR = "#FFD60A"
OTHER_EDGE_COLOR   = "#9A9A9A"

# Area per cell — selectors and all_cells are built 1-to-1
cell_areas = [Cell.SurfaceArea(c) for c in all_cells]
min_area   = min(cell_areas)
max_area   = max(cell_areas)

def area_to_size(area):
    return 12 + int(48 * (area - min_area) / (max_area - min_area + 1))

# Selector positions for nearest-neighbour lookup
sel_pos = [(Vertex.X(s), Vertex.Y(s), Vertex.Z(s)) for s in selectors]

def _d3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** 0.5

def nearest_sel_idx(v):
    p = (Vertex.X(v), Vertex.Y(v), Vertex.Z(v))
    return min(range(len(sel_pos)), key=lambda i: _d3(p, sel_pos[i]))

def nearest_area(v):
    return cell_areas[nearest_sel_idx(v)]

def get_rt(v):
    val = Dictionary.ValueAtKey(Topology.Dictionary(v), "room_type")
    if isinstance(val, list):
        val = val[0] if val else None
    return val

def is_stair_v(v):
    return get_rt(v) == "stairs"

def is_room_v(v):
    return get_rt(v) is not None

all_verts = Graph.Vertices(graph) or []
all_edges = Graph.Edges(graph) or []

room_verts      = [v for v in all_verts if is_room_v(v)]
stair_verts     = [v for v in room_verts if is_stair_v(v)]
non_stair_verts = [v for v in room_verts if not is_stair_v(v)]

# Split stair selectors by plan footprint (X, Y): the main multi-floor shaft shares one
# footprint, while the small entrance stair sits at a different (x, y). Grouping by
# footprint (instead of by object order) is robust to changes in Stair.obj and keeps the
# entrance-stair node sitting on the actual entrance-stair room.
from collections import defaultdict
stair_sel_indices = [i for i, s in enumerate(selectors)
                     if Dictionary.ValueAtKey(Topology.Dictionary(s), "room_type") == "stairs"]
# Cluster stair selectors by plan (X,Y) proximity; largest cluster = main/large stair,
# the rest = entrance stair (robust for the Old Layout's separate stairs).
_XY_TOL_S = 5.0
def _xy_close_s(a, b):
    return ((Vertex.X(selectors[a]) - Vertex.X(selectors[b]))**2
            + (Vertex.Y(selectors[a]) - Vertex.Y(selectors[b]))**2) <= _XY_TOL_S**2
_stair_clusters = []
for i in stair_sel_indices:
    for cl in _stair_clusters:
        if any(_xy_close_s(i, j) for j in cl):
            cl.append(i); break
    else:
        _stair_clusters.append([i])
_chg_s = True
while _chg_s:
    _chg_s = False
    for a in range(len(_stair_clusters)):
        for b in range(a + 1, len(_stair_clusters)):
            if any(_xy_close_s(i, j) for i in _stair_clusters[a] for j in _stair_clusters[b]):
                _stair_clusters[a] += _stair_clusters[b]; del _stair_clusters[b]; _chg_s = True; break
        if _chg_s: break
_stair_clusters.sort(key=len, reverse=True)
main_sel_set      = set(_stair_clusters[0]) if _stair_clusters else set()   # largest cluster = main/large stair
entrance_sel_set  = set(i for cl in _stair_clusters[1:] for i in cl)        # the rest = entrance stair

stair_main_verts = [v for v in stair_verts if nearest_sel_idx(v) in main_sel_set]
stair_ext_verts  = [v for v in stair_verts if nearest_sel_idx(v) in entrance_sel_set]

def _make_stair_rep(verts, label):
    if not verts:
        return None
    sx = sum(Vertex.X(v) for v in verts) / len(verts)
    sy = sum(Vertex.Y(v) for v in verts) / len(verts)
    sz = sum(Vertex.Z(v) for v in verts) / len(verts)
    area = sum(nearest_area(v) for v in verts)
    rep  = Vertex.ByCoordinates(sx, sy, sz)
    rep  = Topology.SetDictionary(rep, Dictionary.ByKeysValues(
        ["v_color", "v_label", "v_size", "room_type"],
        [ROOM_COLOR["stairs"], label, area_to_size(area), "stairs"]
    ))
    return rep

stair_main_rep = _make_stair_rep(stair_main_verts, "stairs")
stair_ext_rep  = _make_stair_rep(stair_ext_verts,  "stairs")

# Style non-stair vertices with area-scaled sizes
vis_verts = []
for v in non_stair_verts:
    rt   = get_rt(v) or "unknown"
    area = nearest_area(v)
    d    = Dictionary.SetValuesAtKeys(
        Topology.Dictionary(v),
        ["v_color", "v_label", "v_size"],
        [ROOM_COLOR.get(rt, "#AAAAAA"), rt, area_to_size(area)]
    )
    vis_verts.append(Topology.SetDictionary(v, d))

if stair_main_rep:
    vis_verts.append(stair_main_rep)
if stair_ext_rep:
    vis_verts.append(stair_ext_rep)

# Build a lookup: for any stair graph vertex, which rep node does it map to?
def stair_rep_for(v):
    if not is_stair_v(v):
        return v
    return stair_main_rep if nearest_sel_idx(v) in main_sel_set else stair_ext_rep

# Edges: remap stair endpoints to their rep node
vis_edges  = []
seen_pairs = set()
for e in all_edges:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    if not is_room_v(sv) or not is_room_v(ev):
        continue
    sv2 = stair_rep_for(sv)
    ev2 = stair_rep_for(ev)
    if sv2 is ev2:
        continue
    key = tuple(sorted([id(sv2), id(ev2)]))
    if key in seen_pairs:
        continue
    seen_pairs.add(key)
    new_e = Edge.ByVertices([sv2, ev2])
    if new_e:
        # Two-colour scheme: yellow if the edge touches a balcony, gray otherwise.
        is_balcony_edge = (get_rt(sv) == "balcony") or (get_rt(ev) == "balcony")
        e_color = BALCONY_EDGE_COLOR if is_balcony_edge else OTHER_EDGE_COLOR
        new_e = Topology.SetDictionary(new_e, Dictionary.ByKeysValues(
            ["e_color"], [e_color]
        ))
        vis_edges.append(new_e)

# Explicitly connect the small (front) balcony to the storeroom in this adjacency view.
# The "small" front balcony is the balcony spatially nearest the storeroom. Deduped by
# rounded endpoint coordinates so it is not drawn twice if the graph already carries it.
def _ckey(v):
    return tuple(round(x, 2) for x in (Vertex.X(v), Vertex.Y(v), Vertex.Z(v)))

_existing_edge_pairs = set()
for e in vis_edges:
    _existing_edge_pairs.add(tuple(sorted([_ckey(Edge.StartVertex(e)), _ckey(Edge.EndVertex(e))])))

store_vis = [v for v in vis_verts if get_rt(v) == "storeroom"]
balc_vis  = [v for v in vis_verts if get_rt(v) == "balcony"]
if store_vis and balc_vis:
    sv_store   = store_vis[0]
    sp         = (Vertex.X(sv_store), Vertex.Y(sv_store), Vertex.Z(sv_store))
    small_balc = min(balc_vis, key=lambda b: _d3((Vertex.X(b), Vertex.Y(b), Vertex.Z(b)), sp))
    pair = tuple(sorted([_ckey(small_balc), _ckey(sv_store)]))
    if pair in _existing_edge_pairs:
        print("Small balcony ↔ storeroom already drawn.")
    else:
        e_bs = Edge.ByVertices([small_balc, sv_store])
        if e_bs:
            e_bs = Topology.SetDictionary(e_bs, Dictionary.ByKeysValues(
                ["e_color"], [BALCONY_EDGE_COLOR]))
            vis_edges.append(e_bs)
            print("Added (vis): small balcony ↔ storeroom")

print(f"Main stair     : {len(stair_main_verts)} cells → 1 node")
print(f"Entrance stair : {len(stair_ext_verts)} cells → 1 node")
print(f"Vis nodes : {len(vis_verts)}  |  Vis edges : {len(vis_edges)}")

Topology.Show(
    vis_verts + vis_edges,
    vertexSizeKey="v_size",
    vertexColorKey="v_color",
    showVertexLabel=True, vertexLabelKey="v_label", vertexLabelFontSize=14,
    edgeWidth=3, edgeColorKey="e_color",
    backgroundColor="white",
    width=900, height=700,
    renderer=renderer
)

Small balcony ↔ storeroom already drawn.
Main stair     : 2 cells → 1 node
Entrance stair : 1 cells → 1 node
Vis nodes : 29  |  Vis edges : 29


## 12. Verify graph vertex and edge dictionaries

In [32]:
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph) or []

print("Sample vertex dictionaries (first 6):")
for v in vertices[:6]:
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    lb = Dictionary.ValueAtKey(d, "label")
    z0 = Dictionary.ValueAtKey(d, "feat_zoning_type_0")
    c0 = Dictionary.ValueAtKey(d, "feat_connectivity_0")
    print(f"  room_type={str(rt):12s}  label={lb}  zoning[0]={z0}  conn[0]={c0}")

print(f"\nSample edge dictionaries (first 5):")
for e in edges[:5]:
    d  = Topology.Dictionary(e)
    dt = Dictionary.ValueAtKey(d, "door_type")
    ks = Dictionary.Keys(d)
    print(f"  door_type={dt}  keys={ks}")

Sample vertex dictionaries (first 6):
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0

Sample edge dictionaries (first 5):
  door_type=door  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'src']
  door_type=door  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'src']
  door_type=door  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'src']
  door_type=door  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 

## 13. Export CSVs in MSD schema

| File | Columns |
|---|---|
| `graphs.csv` | `graph_id`, `num_nodes` |
| `nodes.csv` | `graph_id`, `node_id`, `label`, features, masks |
| `edges.csv` | `graph_id`, `src_id`, `dst_id`, `feat_connectivity_0..2` |

In [33]:
def vkey(v, tol=3):
    return tuple(round(x, tol) for x in Vertex.Coordinates(v))

def gv(d, k, default=0):
    val = Dictionary.ValueAtKey(d, k)
    if isinstance(val, list):
        val = val[0] if val else None
    return val if val is not None else default

# Drop the 13 SelfMerge artifact cells — keep only labelled room cells
labelled = []
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if rt is not None:
        labelled.append((i, v))

print(f"Labelled vertices : {len(labelled)} / {len(vertices)}")

old_to_new   = {old_i: new_i for new_i, (old_i, _) in enumerate(labelled)}
coord_to_new = {vkey(v): old_to_new[old_i] for new_i, (old_i, v) in enumerate(labelled)}

# nodes.csv
nodes_rows = []
for new_i, (old_i, v) in enumerate(labelled):
    d = Topology.Dictionary(v)
    nodes_rows.append({
        "graph_id": 0, "node_id": new_i,
        "label":               int(gv(d, "label", 0)),
        "feat_zoning_type_0":  int(gv(d, "feat_zoning_type_0")),
        "feat_zoning_type_1":  int(gv(d, "feat_zoning_type_1")),
        "feat_zoning_type_2":  int(gv(d, "feat_zoning_type_2")),
        "feat_zoning_type_3":  int(gv(d, "feat_zoning_type_3")),
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
        "train_mask": 0, "val_mask": 0, "test_mask": 1,
    })

# edges.csv — bidirectional (A→B and B→A) to match nx.Graph undirected training format
# only include edges where both endpoints are labelled rooms
edges_rows = []
for e in edges:
    sk  = vkey(Edge.StartVertex(e))
    ek  = vkey(Edge.EndVertex(e))
    src = coord_to_new.get(sk)
    dst = coord_to_new.get(ek)
    if src is None or dst is None:
        continue
    d = Topology.Dictionary(e)
    feat = {
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
    }
    edges_rows.append({"graph_id": 0, "src_id": src, "dst_id": dst, **feat})
    edges_rows.append({"graph_id": 0, "src_id": dst, "dst_id": src, **feat})

pd.DataFrame([{"graph_id": 0, "num_nodes": len(nodes_rows)}]).to_csv(
    os.path.join(DATASET_PATH, "graphs.csv"), index=False)
pd.DataFrame(nodes_rows).to_csv(
    os.path.join(DATASET_PATH, "nodes.csv"), index=False)
pd.DataFrame(edges_rows).to_csv(
    os.path.join(DATASET_PATH, "edges.csv"), index=False)

print(f"graphs.csv : 1 graph")
print(f"nodes.csv  : {len(nodes_rows)} nodes (labelled rooms only)")
print(f"edges.csv  : {len(edges_rows)} rows ({len(edges_rows)//2} doors × 2 directions)")
print(f"Saved to   : {DATASET_PATH}")

Labelled vertices : 30 / 30
graphs.csv : 1 graph
nodes.csv  : 30 nodes (labelled rooms only)
edges.csv  : 64 rows (32 doors × 2 directions)
Saved to   : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B_oldlayout


## 14. Inspect exported CSVs

In [34]:
nodes_df = pd.read_csv(os.path.join(DATASET_PATH, "nodes.csv"))
edges_df = pd.read_csv(os.path.join(DATASET_PATH, "edges.csv"))
print("nodes.csv")
print(nodes_df.to_string(index=False))
print()
print("edges.csv")
print(edges_df.to_string(index=False))

nodes.csv
 graph_id  node_id  label  feat_zoning_type_0  feat_zoning_type_1  feat_zoning_type_2  feat_zoning_type_3  feat_connectivity_0  feat_connectivity_1  feat_connectivity_2  train_mask  val_mask  test_mask
        0        0      0                   1                   0                   0                   0                    0                    1                    0           0         0          1
        0        1      0                   1                   0                   0                   0                    0                    1                    0           0         0          1
        0        2      0                   1                   0                   0                   0                    0                    1                    0           0         0          1
        0        3      0                   1                   0                   0                   0                    0                    1                    0           0  

## 15. Load dataset into PyG

In [35]:
pyg = PyG.ByCSVPath(
    path=DATASET_PATH,
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
)
print(pyg)

## 16. Load the pretrained MSD node classifier

In [36]:
pyg.LoadModel(MODEL_PATH)
print("Model loaded.")

Model loaded.


## 17. Predict room types

In [37]:
def to_class(val):
    a = np.squeeze(np.asarray(val))
    if a.ndim == 0: return int(a)
    if a.ndim == 1: return int(np.argmax(a)) if a.size > 1 else int(a[0])
    raise ValueError(f"Unexpected shape {a.shape}")

report    = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
pred_list = report["pred"]
true_list = report["y_true"]
prob_list = report.get("prob", None)

LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}

rows = []
for g_idx, data in enumerate(pyg.data_list):
    gid   = int(data.graph_id.item()) if hasattr(data, "graph_id") else g_idx
    n     = data.num_nodes
    gp    = np.asarray(pred_list[g_idx])
    gt    = np.asarray(true_list[g_idx])
    gprob = np.asarray(prob_list[g_idx]) if prob_list else None
    for ni in range(n):
        yt  = to_class(gt[ni])
        yp  = to_class(gp[ni])
        row = {"graph_id": gid, "node_id": ni, "y_true": yt, "y_pred": yp}
        if gprob is not None:
            p = np.squeeze(np.asarray(gprob[ni]))
            if p.ndim == 1 and yp < p.size:
                row["y_pred_prob"] = float(p[yp])
        rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df["true_name"] = pred_df["y_true"].map(LABEL_NAME)
pred_df["pred_name"] = pred_df["y_pred"].map(LABEL_NAME)

PRED_CSV = os.path.join(DATASET_PATH, "node_predictions.csv")
pred_df.to_csv(PRED_CSV, index=False)

correct = (pred_df["y_true"] == pred_df["y_pred"]).sum()
total   = len(pred_df)
print(f"Predictions: {correct}/{total} correct = {correct/total:.1%}")
print(f"Saved: {PRED_CSV}")
print(pred_df[["node_id","true_name","pred_name"]].to_string(index=False))

Predictions: 15/30 correct = 50.0%
Saved: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B_oldlayout\node_predictions.csv
 node_id  true_name pred_name
       0    bedroom   bedroom
       1    bedroom   bedroom
       2    bedroom   bedroom
       3    bedroom   bedroom
       4    bedroom   bedroom
       5    bedroom   bedroom
       6    bedroom   bedroom
       7 livingroom   kitchen
       8 livingroom   kitchen
       9 livingroom  corridor
      10    kitchen  corridor
      11   corridor   kitchen
      12   corridor   kitchen
      13   corridor   kitchen
      14   corridor   kitchen
      15   corridor   kitchen
      16     stairs storeroom
      17     stairs    stairs
      18     stairs    stairs
      19   bathroom  bathroom
      20   bathroom storeroom
      21  storeroom storeroom
      22  storeroom  bathroom
      23  storeroom  bathroom
      24  storeroom storeroom
      25  storeroom  bathroom
      26  storeroom storeroom
      27    balcony   balc

## 18. Visualise true vs predicted labels

Misclassified nodes shown in **red** (size 30).  
Correctly classified nodes use a colour scale.

In [38]:
LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}

pred_lookup  = pred_df.set_index("node_id")[["y_true", "y_pred"]].to_dict("index")

# Style only the 23 labelled vertices — artifact cells are never added to vis_verts
vis_verts   = []
new_i_to_v  = {}
for new_i, (old_i, v) in enumerate(labelled):
    row = pred_lookup.get(new_i, {"y_true": 0, "y_pred": 0})
    yt  = int(row["y_true"])
    yp  = int(row["y_pred"])
    d   = Topology.Dictionary(v)
    if yt != yp:
        sz = 30
        tc = pc = "red"
    else:
        sz = 14
        tc = Color.ByValueInRange(yt, minValue=0, maxValue=8)
        pc = Color.ByValueInRange(yp, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(
        d,
        ["true_color", "pred_color", "node_size", "true_label", "pred_label"],
        [tc, pc, sz, LABEL_NAME.get(yt, str(yt)), LABEL_NAME.get(yp, str(yp))]
    )
    v = Topology.SetDictionary(v, d)
    vis_verts.append(v)
    new_i_to_v[new_i] = v

# Build edges directly between labelled vertices (skips artifact cells entirely)
vis_edges = []
seen_epairs = set()
for e in edges:
    sk  = vkey(Edge.StartVertex(e))
    ek  = vkey(Edge.EndVertex(e))
    src = coord_to_new.get(sk)
    dst = coord_to_new.get(ek)
    if src is None or dst is None:
        continue
    key = tuple(sorted([src, dst]))
    if key in seen_epairs:
        continue
    seen_epairs.add(key)
    new_e = Edge.ByVertices([new_i_to_v[src], new_i_to_v[dst]])
    if new_e:
        vis_edges.append(new_e)

correct = sum(1 for r in pred_lookup.values() if r["y_true"] == r["y_pred"])
print(f"Correct: {correct}/{len(vis_verts)}  Isolated node 13 (bathroom) has no edges — expected.")

print("--- True labels ---")
Topology.Show(
    vis_verts + vis_edges,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="true_color",
    showVertexLabel=True, vertexLabelKey="true_label", vertexLabelFontSize=14,
    edgeWidth=2,
    backgroundColor="white",
    width=900, height=600, renderer=renderer
)

print("--- Predicted labels ---")
Topology.Show(
    vis_verts + vis_edges,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="pred_color",
    showVertexLabel=True, vertexLabelKey="pred_label", vertexLabelFontSize=14,
    edgeWidth=2,
    backgroundColor="white",
    width=900, height=600, renderer=renderer
)

Correct: 15/30  Isolated node 13 (bathroom) has no edges — expected.
--- True labels ---


--- Predicted labels ---
